# Week 6: Data Retrieval and Processing
This notebook implements the data retrieval and preprocessing pipeline for Milestone 1 IoT data stored on the blockchain, as specified in the [docs/Code Template.md](file:///Users/brianjancarlos/codestuff/MMDC/ADET/adet/docs/Code%20Template.md).

Reference: [MS1_Smart_Tracking_System_Blockchain_Ledger_Submission_TeamKaizen.ipynb](file:///Users/brianjancarlos/codestuff/MMDC/ADET/adet/MS1_Smart_Tracking_System_Blockchain_Ledger_Submission_TeamKaizen.ipynb)

In [1]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from web3 import Web3

def get_env_value(key, default=None, env_path=".env"):
    # Prefer shell environment variables, fallback to local .env
    value = os.getenv(key)
    if value:
        return value

    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8") as env_file:
            for line in env_file:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                name, raw_value = line.split("=", 1)
                if name.strip() == key:
                    return raw_value.strip().strip('"').strip("'")
    return default

def get_env_int(key, default):
    return int(get_env_value(key, str(default)))

def get_env_float(key, default):
    return float(get_env_value(key, str(default)))

# Connect to local Ganache blockchain
ganache_url = get_env_value("GANACHE_URL", "http://127.0.0.1:8545")
web3 = Web3(Web3.HTTPProvider(ganache_url))

if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [2]:
# Load deployed smart contract configuration
contract_address = get_env_value("CONTRACT_ADDRESS")
if not contract_address:
    raise ValueError("CONTRACT_ADDRESS is missing. Set it in .env or the environment.")
contract_address = Web3.to_checksum_address(contract_address)

abi_path = Path(get_env_value("ABI_PATH", "contracts/abi.json"))

# Load ABI
with open(abi_path, "r", encoding="utf-8") as abi_file:
    abi = json.load(abi_file)

# Instantiate the contract
contract = web3.eth.contract(address=contract_address, abi=abi)

# Configure default account
contract_owner = contract.functions.owner().call()
if contract_owner not in web3.eth.accounts:
    override_owner = get_env_value("CONTRACT_OWNER")
    if override_owner:
        contract_owner = Web3.to_checksum_address(override_owner)
    else:
        contract_owner = web3.eth.accounts[0]

web3.eth.default_account = contract_owner

print(f"✅ Connected to Smart Contract at {contract_address}")
print(f"✅ Using default sender account: {web3.eth.default_account}")

✅ Connected to Smart Contract at 0x7abf4b356FB67C8a9917c7E1E543895DB1Bf53b4
✅ Using default sender account: 0x1C73Dd704ffeE88a4f4aAD5bA3B1af87C5884D0F


In [3]:
# Get the total number of stored records from blockchain
total_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {total_records}")

# Retrieve and print the first stored record to verify retrieval works
if total_records > 0:
    first_record = contract.functions.getRecord(0).call()
    print("First Stored Record:", first_record)
else:
    print("No records stored yet on the blockchain.")

Total IoT records stored: 265
First Stored Record: [1780192485, 'PKG7545', 'Location', 'Naha Central Post Office']


In [4]:
# Fetch all stored IoT data and structure it in a DataFrame
data = []
for i in range(total_records):
    record = contract.functions.getRecord(i).call()
    data.append({
        "timestamp": record[0],
        "device_id": record[1],
        "data_type": record[2],
        "data_value": record[3]
    })

# Convert to a DataFrame
df = pd.DataFrame(data)

# Convert timestamp to readable format
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")

# Display first few records
print("Raw retrieved blockchain data preview:")
display(df.head())

Raw retrieved blockchain data preview:


,timestamp,device_id,data_type,data_value
0,2026-05-31 01:54:45,PKG7545,Location,Naha Central Post Office
1,2026-05-31 01:54:45,PKG7545,Status,Out for Delivery
2,2026-05-31 01:54:46,PKG2659,Location,Nagoya Central Post Office
3,2026-05-31 01:54:46,PKG2659,Status,Arrival
4,2026-05-31 01:54:46,PKG7965,Location,Nagoya Central Post Office


In [5]:
# Data Preprocessing and Cleaning
print("Identifying missing values before cleaning:")
print(df.isna().sum())

# Extract numerical values from 'data_value'
# Note: Improved regex r'(-?\d+\.?\d*)' is used instead of template r'(\d+\.?\d*)' 
# to correctly capture negative numbers such as negative temperatures.
df["numeric_value"] = df["data_value"].str.extract(r'(-?\d+\.?\d*)').astype(float)

# Handle missing values (if any)
# Fill missing numeric values with the median of the column (if significant) or 0 (if minor)
# The template suggests using fillna(0) for minor missing values.
df.fillna(0, inplace=True)

# Display cleaned data
print("\nCleaned and preprocessed data preview:")
display(df.head())

Identifying missing values before cleaning:
timestamp     0
device_id     0
data_type     0
data_value    0
dtype: int64

Cleaned and preprocessed data preview:


,timestamp,device_id,data_type,data_value,numeric_value
0,2026-05-31 01:54:45,PKG7545,Location,Naha Central Post Office,0.0
1,2026-05-31 01:54:45,PKG7545,Status,Out for Delivery,0.0
2,2026-05-31 01:54:46,PKG2659,Location,Nagoya Central Post Office,0.0
3,2026-05-31 01:54:46,PKG2659,Status,Arrival,0.0
4,2026-05-31 01:54:46,PKG7965,Location,Nagoya Central Post Office,0.0


In [6]:
# Save cleaned IoT data to a CSV file in the assets/ directory
output_path = "assets/cleaned_iot_data.csv"
df.to_csv(output_path, index=False)
print(f"✅ Cleaned IoT data saved successfully as {output_path}")

# Create another copy of it named "MO-IT148 Homework: Data Retrieval and Processing S3101 Team Kaizen.csv" in assets folder
homework_output_path = "assets/MO-IT148 Homework: Data Retrieval and Processing S3101 Team Kaizen.csv"
df.to_csv(homework_output_path, index=False)
print(f"✅ Homework copy saved successfully as {homework_output_path}")

✅ Cleaned IoT data saved successfully as assets/cleaned_iot_data.csv
✅ Homework copy saved successfully as assets/MO-IT148 Homework: Data Retrieval and Processing S3101 Team Kaizen.csv


## Enriched Logistics Dataset & Cold-Chain KPIs
This section merges the blockchain retrieved log records back with the original CSV file to enrich the dataset with context (such as `origin`, `delivery_location`, and `perishable` status), formats it to match wide logistics tables, and calculates temperature issue alerts.

In [7]:
# Load CSV file path from environment for data enrichment
csv_path = Path(get_env_value("CSV_PATH", "IOT Data Simulation/smart_logistic_tracker_japan.csv"))

# Load original CSV data
try:
    df_csv = pd.read_csv(csv_path)
    df_csv.columns = [col.strip() for col in df_csv.columns]
except Exception as e:
    print(f"⚠️ Could not load CSV for enrichment: {e}")
    df_csv = pd.DataFrame()

# Pivot the retrieved blockchain data to wide format
df_wide = df.pivot_table(
    index=["timestamp", "device_id"], 
    columns="data_type", 
    values=["data_value", "numeric_value"], 
    aggfunc="first"
)

# Flatten columns naming from MultiIndex
df_wide.columns = [f"{col[1]}_{col[0]}".lower() for col in df_wide.columns]
df_wide = df_wide.reset_index()

# Rename fields for cleaner logistics readability
rename_cols = {
    "status_data_value": "status",
    "temperature_numeric_value": "temperature_celsius",
    "temperature_data_value": "temperature",
    "humidity_numeric_value": "humidity",
    "location_data_value": "current_location"
}
df_wide = df_wide.rename(columns={k: v for k, v in rename_cols.items() if k in df_wide.columns})
df_wide = df_wide.rename(columns={"timestamp": "blockchain_timestamp"})

# Merge wide-format blockchain data with original CSV context
if not df_csv.empty:
    is_iot = "shipment_id" in df_csv.columns
    csv_key = "shipment_id" if is_iot else "package_id"
    
    # Merge datasets
    df_enriched = pd.merge(
        df_wide,
        df_csv,
        left_on="device_id",
        right_on=csv_key,
        suffixes=("_blockchain", "_csv")
    )
    
    # Resolve column name collisions (e.g. temperature, humidity, status)
    for col in ["temperature", "humidity", "status", "current_location"]:
        blockchain_col = f"{col}_blockchain"
        if blockchain_col in df_enriched.columns:
            df_enriched[col] = df_enriched[blockchain_col]
        elif f"{col}_csv" in df_enriched.columns and col not in df_enriched.columns:
            df_enriched[col] = df_enriched[f"{col}_csv"]
            
    # Also resolve temperature_celsius if it matches numeric column directly
    if "temperature_celsius" not in df_enriched.columns and "temperature_numeric_value" in df_enriched.columns:
         df_enriched["temperature_celsius"] = df_enriched["temperature_numeric_value"]
    elif "temperature_celsius" not in df_enriched.columns and "temperature_csv" in df_enriched.columns:
         df_enriched["temperature_celsius"] = df_enriched["temperature_csv"]
    
    # Calculate temperature_issue if 'perishable' exists
    if "perishable" in df_enriched.columns and "temperature_celsius" in df_enriched.columns:
        def get_temp_issue(row):
            is_perish = str(row["perishable"]).strip().lower() in ["yes", "true", "1"]
            if not is_perish:
                return "Not Applicable"
            temp = row["temperature_celsius"]
            # Safe cold chain range: -2.0 to 15.0
            if -2.0 <= temp <= 15.0:
                return "Normal"
            else:
                return "Temperature Issue"
        
        df_enriched["temperature_issue"] = df_enriched.apply(get_temp_issue, axis=1)
    
    # Reorder and filter columns to match the target logistics template format
    target_cols_order = [
        "blockchain_timestamp", "device_id", "order_date", "delivery_date", 
        "origin", "current_location", "delivery_location", "temperature", 
        "temperature_celsius", "perishable", "temperature_issue", "status"
    ]
    final_cols = [col for col in target_cols_order if col in df_enriched.columns]
    
    # Append any additional relevant columns from original CSV
    for extra in ["carrier", "tracking_number", "waiting_time", "traffic_status", "tamper_alert"]:
        if extra in df_enriched.columns and extra not in final_cols:
            final_cols.append(extra)
            
    df_display = df_enriched[final_cols].rename(
        columns={"blockchain_timestamp": "timestamp", "device_id": "package_id"}
    )
    
    print("📊 Enriched Logistics Dataset (Wide Format):")
    display(df_display.head())
    
    # Save enriched dataset as a secondary CSV
    enriched_output_path = "assets/enriched_logistics_data.csv"
    df_display.to_csv(enriched_output_path, index=False)
    print(f"✅ Enriched logistics dataset saved successfully as {enriched_output_path}")
else:
    print("Enrichment skipped. Showing wide-format retrieved data:")
    display(df_wide.head())


📊 Enriched Logistics Dataset (Wide Format):


,timestamp,package_id,current_location,temperature,temperature_celsius,status,waiting_time,traffic_status,tamper_alert
0,2026-06-05 10:34:02,SHP4147,NaN,NaN,NaN,Out for Delivery,75,Clear,No
1,2026-06-05 10:34:03,SHP4147,NaN,20.8°C,20.8,NaN,75,Clear,No
2,2026-06-05 10:34:05,SHP4147,NaN,NaN,NaN,NaN,75,Clear,No
3,2026-06-05 10:34:07,SHP6541,NaN,NaN,NaN,In Transit,33,Detour,Yes
4,2026-06-05 10:34:08,SHP6541,NaN,17.3°C,17.3,NaN,33,Detour,Yes


✅ Enriched logistics dataset saved successfully as assets/enriched_logistics_data.csv
